In [ ]:
import os
import subprocess
import sys
from pathlib import Path

print("====== PREPARING NATURAL-CORRUPTION BENCHMARK AND APRIL-GAN ======")
BENCHMARK_REPOSITORY = "Parsagh05/Natural-Corruption-Robustness"
BENCHMARK_ROOT = Path("/kaggle/working/Natural-Corruption-Robustness")
APRILGAN_ROOT = Path("/kaggle/working/VAND-APRIL-GAN")
APRILGAN_COMMIT = "f13b8a634e04f9fde8fa03db125b25af5695d8e1"

if not BENCHMARK_ROOT.exists():
    subprocess.run(
        ["git", "clone", f"https://github.com/{BENCHMARK_REPOSITORY}.git", str(BENCHMARK_ROOT)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(BENCHMARK_ROOT), "pull", "--ff-only"], check=True
    )

required_wrapper = BENCHMARK_ROOT / "few_shot/harness/models.py"
if not required_wrapper.is_file() or "APRILGANFewShotWrapper" not in required_wrapper.read_text(encoding="utf-8"):
    raise RuntimeError(
        "The cloned benchmark revision does not yet contain APRIL-GAN support. "
        "Commit and push these local changes before running on Kaggle."
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--disable-pip-version-check",
        "-r",
        str(BENCHMARK_ROOT / "few_shot/requirements.txt"),
    ],
    check=True,
)

if not APRILGAN_ROOT.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/ByChelsea/VAND-APRIL-GAN.git", str(APRILGAN_ROOT)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(APRILGAN_ROOT), "fetch", "origin"], check=True
    )
subprocess.run(
    ["git", "-C", str(APRILGAN_ROOT), "checkout", "--detach", APRILGAN_COMMIT],
    check=True,
)
resolved_commit = subprocess.run(
    ["git", "-C", str(APRILGAN_ROOT), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if resolved_commit != APRILGAN_COMMIT:
    raise RuntimeError(f"Wrong APRIL-GAN source commit: {resolved_commit}")

required_upstream = [
    APRILGAN_ROOT / "open_clip" / "factory.py",
    APRILGAN_ROOT / "model.py",
    APRILGAN_ROOT / "prompt_ensemble.py",
    APRILGAN_ROOT / "exps" / "pretrained" / "mvtec_pretrained.pth",
    APRILGAN_ROOT / "exps" / "pretrained" / "visa_pretrained.pth",
]
missing_upstream = [str(path) for path in required_upstream if not path.is_file()]
if missing_upstream:
    raise FileNotFoundError(
        "APRIL-GAN clone/checkpoints are incomplete:\n  - " + "\n  - ".join(missing_upstream)
    )

for import_path in (BENCHMARK_ROOT, APRILGAN_ROOT):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))
os.environ["APRILGAN_ROOT"] = str(APRILGAN_ROOT)

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running APRIL-GAN.")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Official APRIL-GAN commit: {resolved_commit}")
print("Released cross-dataset checkpoints found in exps/pretrained.")


# APRIL-GAN few-shot natural-corruption benchmark

APRIL-GAN does not publish separate few-shot checkpoints. This notebook uses the same released cross-dataset projection checkpoint as zero-shot inference and builds a clean target-normal memory bank for each category. It reproduces the official `torch.randint` support sampling (including replacement), four-layer nearest-neighbour distance maps, and class/condition image-score fusion. Support images stay clean while only test images receive the configured corruptions.


In [ ]:
import gc

from few_shot.harness.dataset import build_dataset_configs
from few_shot.harness.models import APRILGANFewShotWrapper, discover_aprilgan_checkpoints
from few_shot.harness.runner import run_aprilgan_evaluations

DATASET_NAME = "visa"  # "mvtec", "visa", or "both"
DATASET_NAME = DATASET_NAME.lower().strip()
if DATASET_NAME not in {"mvtec", "visa", "both"}:
    raise ValueError("DATASET_NAME must be 'mvtec', 'visa', or 'both'.")
DATASETS_TO_RUN = ("mvtec", "visa") if DATASET_NAME == "both" else (DATASET_NAME,)

MVTEC_ROOT = "/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection"
VISA_ROOT = "/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922"
OUTPUT_ROOT = "/kaggle/working/outputs"
CLIP_DOWNLOAD_DIR = "/kaggle/working/aprilgan-clip"
clip_candidates = []
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    clip_candidates.extend(kaggle_input.rglob("ViT-L-14-336px.pt"))
CLIP_WEIGHT_PATH = str(next((path for path in clip_candidates if path.is_file()), ""))

USE_CATEGORIZED_CORRUPTIONS = True
CORRUPTION_SEED = 123
UNCATEGORIZED_CORRUPTION_TYPES = [
    "gaussian_noise", "shot_noise", "impulse_noise", "defocus_blur",
    "motion_blur", "zoom_blur", "brightness", "contrast",
]
CATEGORIZED_CORRUPTION_TYPES = ["noise", "blur", "photometric", "geometric"]
CORRUPTION_TYPES = (
    CATEGORIZED_CORRUPTION_TYPES
    if USE_CATEGORIZED_CORRUPTIONS
    else UNCATEGORIZED_CORRUPTION_TYPES
)
INCLUDE_CLEAN_BASELINE = True
SEVERITY_LEVELS = [1, 2, 3, 4]
SHOTS_TO_RUN = [1, 2, 4]
REFERENCE_SEEDS = [42]
BATCH_SIZE = 1
DEVICE = "cuda"
CORRUPTION_CACHE_ROOT = None
CORRUPTION_CACHE_FORMAT = "png"

CHECKPOINT_PATHS = discover_aprilgan_checkpoints(
    str(APRILGAN_ROOT / "exps" / "pretrained")
)
dataset_configs = build_dataset_configs(
    mvtec_root=MVTEC_ROOT if "mvtec" in DATASETS_TO_RUN else None,
    visa_root=VISA_ROOT if "visa" in DATASETS_TO_RUN else None,
)
config_by_dataset = {
    ("mvtec" if config.name.lower().startswith("mvtec") else "visa"): config
    for config in dataset_configs
}
missing = [name for name in DATASETS_TO_RUN if name not in config_by_dataset]
if missing:
    raise FileNotFoundError(f"Could not resolve selected Kaggle dataset roots: {missing}")
resolved_roots = {
    name: str(config_by_dataset[name].root_path) for name in DATASETS_TO_RUN
}

# Preflight the largest requested support selection before loading the CLIP model.
support_probe = APRILGANFewShotWrapper(
    checkpoint_paths=CHECKPOINT_PATHS,
    dataset_roots=resolved_roots,
    shot=max(SHOTS_TO_RUN),
    reference_seed=REFERENCE_SEEDS[0],
    device=DEVICE,
)
for dataset_name in DATASETS_TO_RUN:
    selections = support_probe._select_support_paths(dataset_name)
    print(
        f"Support preflight: {dataset_name}, {len(selections)} categories, "
        f"{max(SHOTS_TO_RUN)} draws/category."
    )

print("LAUNCHING APRIL-GAN FEW-SHOT ROBUSTNESS BENCHMARK")
print(f"Datasets: {DATASETS_TO_RUN}; shots={SHOTS_TO_RUN}; support seeds={REFERENCE_SEEDS}")
print("Checkpoints: released cross-dataset exps/pretrained files")
print(f"CLIP backbone: {CLIP_WEIGHT_PATH or 'official download'}")
print(f"Corruptions: {CORRUPTION_TYPES} @ {SEVERITY_LEVELS}; clean={INCLUDE_CLEAN_BASELINE}")

run_aprilgan_evaluations(
    mvtec_root=MVTEC_ROOT,
    visa_root=VISA_ROOT,
    output_root=OUTPUT_ROOT,
    aprilgan_root=str(APRILGAN_ROOT),
    checkpoint_paths=CHECKPOINT_PATHS,
    shots=SHOTS_TO_RUN,
    datasets=DATASETS_TO_RUN,
    reference_seeds=REFERENCE_SEEDS,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    corruption_types=CORRUPTION_TYPES,
    severity_levels=SEVERITY_LEVELS,
    categorized_corruptions=USE_CATEGORIZED_CORRUPTIONS,
    corruption_cache_root=CORRUPTION_CACHE_ROOT,
    corruption_cache_format=CORRUPTION_CACHE_FORMAT,
    corruption_seed=CORRUPTION_SEED,
    include_clean=INCLUDE_CLEAN_BASELINE,
    strict_source_commit=True,
    clip_weight_path=CLIP_WEIGHT_PATH,
    clip_download_dir=CLIP_DOWNLOAD_DIR,
)

gc.collect()
torch.cuda.empty_cache()
archives = []
for shot in SHOTS_TO_RUN:
    for seed in REFERENCE_SEEDS:
        suffix = f"-seed-{seed}" if len(REFERENCE_SEEDS) > 1 else ""
        archives.append(f"APRIL-GAN-{shot}-shot{suffix}_artifacts.zip")
print(f"Finished. Collect from {OUTPUT_ROOT}: {archives}")
